# 04. Tensor Parallelism Megatron Sequence and Async TP

- **Category:** 16_distributed_and_scalable
- **Lineage & Citations:** Shoeybi et al. (Megatron-LM, 2019, *Efficient Large-Scale Language Model Training on GPU Clusters*), Korthikanti et al. (Sequence Parallelism, 2022, *Reducing Activation Recomputation in Large Transformer Models*), Megatron-Core (Async TP, 2023).
- **Purpose:** Decompose intra-layer GEMMs across column- and row-parallel ranks with fused activations, and eliminate non-TP activation redundancies via sequence sharding.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [ZeRO Stages 1, 2, 3 and FSDP2 Fully Shard](03_zero_stages_1_2_3_and_fsdp2_fully_shard.ipynb) $\to$ `🔗 Downstream Syntheses:` [Context Parallelism Ring Attention and Ulysses](05_context_parallelism_ring_attention_and_ulysses.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use Tensor Parallelism across slow inter-node networks (e.g. standard InfiniBand or Ethernet without NVLink). Intra-layer TP requires multiple AllReduce/ReduceScatter collectives per layer, inducing crippling latency stalls unless confined to high-bandwidth NVSwitch/NVLink domains (TP $\le 8$).

## Mathematical Foundations & Governing Equations

$$
Y = \sum_{r=0}^{W-1} \sigma\left( X W_1^{(r)} \right) W_2^{(r)}, \quad T_{\text{TP}} = 2 \frac{W-1}{W} S \beta, \quad T_{\text{SP}} = 2 \frac{W-1}{W} S \beta
$$

### Asymptotic Time & Memory Complexity
- **Computation Scaling:** Column and row projections scale as $O\left(\frac{B \cdot S \cdot D \cdot H}{W}\right)$ FLOPs per rank.
- **Sequence Parallel Activation Footprint:** Reduces non-TP activation memory (LayerNorm, Dropout) from $O(B \cdot S \cdot D)$ to $O\left(\frac{B \cdot S \cdot D}{W}\right)$ with identical communication volume.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def megatron_tp_mlp(x: torch.Tensor, w1_shards: torch.Tensor, w2_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    world_size = w1_shards.shape[0]
    partials = []
    for r in range(world_size):
        h_r = F.gelu(x @ w1_shards[r])
        y_r = h_r @ w2_shards[r]
        partials.append(y_r)
    partial_shards = torch.stack(partials, dim=0)
    y = torch.sum(partial_shards, dim=0)
    return y, partial_shards

def sequence_parallel_mlp(x_seq_shards: torch.Tensor, w1_shards: torch.Tensor, w2_shards: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    world_size = x_seq_shards.shape[0]
    x_full = torch.cat([x_seq_shards[r] for r in range(world_size)], dim=1)
    y_full, _ = megatron_tp_mlp(x_full, w1_shards, w2_shards)
    chunks = torch.chunk(y_full, world_size, dim=1)
    y_seq_shards = torch.stack(chunks, dim=0)
    return y_seq_shards, x_full

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

tp_out, partials = megatron_tp_mlp(weights["x"], weights["w1_shards"], weights["w2_shards"])
assert float(torch.max(torch.abs(tp_out - weights["tp_mlp_out"]))) < 1e-5
assert float(torch.max(torch.abs(partials - weights["partial_shards"]))) < 1e-5

sp_out, x_full = sequence_parallel_mlp(weights["x_seq_shards"], weights["w1_shards"], weights["w2_shards"])
assert float(torch.max(torch.abs(sp_out - weights["sp_mlp_out_shards"]))) < 1e-5
assert float(torch.max(torch.abs(x_full - weights["x_full"]))) < 1e-5
print("All Megatron TP and Sequence Parallelism parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: megatron_tp_mlp(weights["x"], weights["w1_shards"], weights["w2_shards"]), iters=50, name="Megatron TP MLP"),
    benchmark_fn(lambda: sequence_parallel_mlp(weights["x_seq_shards"], weights["w1_shards"], weights["w2_shards"]), iters=50, name="Sequence Parallel MLP"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Replicated Non-TP Activations vs Sequence Parallelism Sharding
batch = 4
seq_len = 8192
hidden_dim = 4096
world_size = 8

# Standard Megatron TP replicates LayerNorm/Dropout input across all world_size ranks
tp_replicated_bytes = batch * seq_len * hidden_dim * 2  # FP16
tp_replicated_mb = tp_replicated_bytes / (1024 ** 2)

# Sequence Parallelism shards sequence across ranks: s / world_size
sp_sharded_bytes = batch * (seq_len // world_size) * hidden_dim * 2
sp_sharded_mb = sp_sharded_bytes / (1024 ** 2)

print(f"Megatron TP Non-Parallel Activation per GPU: {tp_replicated_mb:.2f} MB")
print(f"Sequence Parallel Activation per GPU:          {sp_sharded_mb:.2f} MB  ({tp_replicated_mb / sp_sharded_mb:.1f}x reduction)")
assert tp_replicated_mb == world_size * sp_sharded_mb, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Conjugate Column-Row Composition:** Pairing column-parallel projections with row-parallel projections confines communication to exactly one AllReduce per MLP and MHA block.
2. **Sequence Parallelism Communication Invariance:** Replacing monolithic AllReduce with distributed AllGather and ReduceScatter primitives reduces activation footprint by $W\times$ at zero communication penalty.
3. **Physical Domain Boundary:** Tensor Parallelism requires ultra-dense interconnects ($> 900\text{ GB/s}$ NVLink); across nodes, prefer Data Parallelism or Pipeline Parallelism.

---
### Cross-Domain Generalisation Matrix
| Domain | Intra-Operator Decomposition Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Parallel Fast Fourier Transforms (2D/3D FFT)** | Pencil/slab transposition across mesh ranks | Eliminates all-to-all bottleneck via 1D line transforms |
| **Quantum Circuit Simulation** | State-vector tensor slicing across QPU partitions | Accommodates $2^N$ Hilbert state amplitudes across distributed nodes |
| **Atmospheric Climate Models (WRF)** | Latitude-longitude domain pencil decomposition | Balances boundary ghost-cell communication against stencil depth |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [04_tensor_parallelism_megatron_sequence_and_async_tp.md](../../markdown/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.md)
- **PyTorch 2.x Module:** [04_tensor_parallelism_megatron_sequence_and_async_tp.py](../../pytorch2/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.py)
- **Functional JAX Module:** [04_tensor_parallelism_megatron_sequence_and_async_tp.py](../../jax/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.py)
- **Native Rust Candle Track:** [04_tensor_parallelism_megatron_sequence_and_async_tp.rs](../../candle/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.rs)
- **Hardware Kernel (CPU / WGSL):** [04_tensor_parallelism_megatron_sequence_and_async_tp.rs](../../kernels/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.rs)
- **Golden SafeTensors:** [04_tensor_parallelism_megatron_sequence_and_async_tp.safetensors](../../weights/16_distributed_and_scalable/04_tensor_parallelism_megatron_sequence_and_async_tp.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 03. ZeRO Stages 1, 2, 3 and FSDP2 Fully Shard](03_zero_stages_1_2_3_and_fsdp2_fully_shard.ipynb) | [Master Index](../../README.md#16_distributed_and_scalable) | [Next: 05. Context Parallelism Ring Attention and Ulysses](05_context_parallelism_ring_attention_and_ulysses.ipynb) &rarr;